In [0]:
#Bibliotecas 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# #Tratamento das tabelas
 
# PARQUET_PATH = "/Volumes/workspace/olist_bronze/parquet"

# tables = [
#     "olist_customers_dataset",
#     "olist_orders_dataset",
#     "olist_order_items_dataset",
#     "olist_order_payments_dataset",
#     "olist_order_reviews_dataset",
#     "olist_products_dataset",
#     "olist_sellers_dataset",
#     "olist_geolocation_dataset",
#     "product_category_name_translation"
# ]

# dataframes = {}

# for table in tables:
#     df = spark.read.parquet(f"{PARQUET_PATH}/{table}")
#     dataframes[table] = df

# # Diagnóstico de valores nulos
# null_results = []

# for table, df in dataframes.items():

#     total_rows = df.count()

#     for column in df.columns:

#         null_count = df.filter(
#             F.col(column).isNull()
#         ).count()

#         null_results.append((
#             table,
#             column,
#             total_rows,
#             null_count,
#             round(null_count / total_rows * 100, 2)
#             if total_rows > 0 else 0
#         ))

# null_report = spark.createDataFrame(
#     null_results,
#     ["table_name", "column_name", "total_rows",
#      "null_count", "null_percentage"]
# )

# display(
#     null_report
#     .filter(F.col("null_count") > 0)
#     .orderBy(F.desc("null_percentage"))
# )


In [0]:
""" 
Nulos esperados na base de dados:
Tabela/coluna	                        Nulos	Interpretação	
reviews.review_comment_title	        88,47%	Cliente não informou título	
reviews.review_comment_message	        60,56%	Cliente não escreveu comentário	
orders.order_delivered_customer_date	2,98%	Pedido pode não ter sido entregue	
orders.order_delivered_carrier_date	    1,79%	Pedido pode não ter sido enviado	
orders.order_approved_at	            0,16%	Aprovação ausente	
products.product_category_name	        1,85%	Produto sem categoria informada
"""

In [0]:
# Registra as tabelas como views temporárias
# for table, df in dataframes.items():
#     df.createOrReplaceTempView(table)

# # Excluir os registros que tiver o review_creation_date NULL (8.764 registros)
# display(
#     spark.sql("""
#         SELECT COUNT(*)
#         FROM olist_order_reviews_dataset
#         WHERE review_creation_date IS NULL
#     """)
# )

# Excluir os registros quando order_status = 'delivered' AND order_delivered_customer_date IS NULL (8 registros)
# display(
#     spark.sql("""
#         SELECT *
#         FROM olist_orders_dataset
#         WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL
#     """)
# )

# Excluir os registros que tiver o product_category_name NULL (610 registros)
# display(
#     spark.sql("""
#         SELECT *
#         FROM olist_products_dataset
#         WHERE product_category_name IS NULL
#     """)
# )

# Excluir os registros que tiver review_id NULL (1 registro)
# display(
#     spark.sql("""
#         SELECT *
#         FROM olist_order_reviews_dataset
#         WHERE review_id IS NULL
#     """)
# )

# Excluir os registros que tiver order_id NULL (2.236 registros)
# display(
#     spark.sql("""
#         SELECT *
#         FROM olist_order_reviews_dataset
#         WHERE order_id IS NULL
#     """)
# )

# Excluir os registros que tiver review_answer_timestamp NULL (20 registros)
# display(
#     spark.sql("""
#         SELECT *
#         FROM olist_order_reviews_dataset
#         WHERE review_answer_timestamp IS NULL
#     """)
# )

In [0]:
# ============================================================
# CONFIGURAÇÕES
# ============================================================

BRONZE_PATH = "/Volumes/workspace/olist_bronze/parquet"
SILVER_PATH = "/Volumes/workspace/olist_silver/parquet"

TABELAS = {
    "orders": "olist_orders_dataset",
    "customers": "olist_customers_dataset",
    "items": "olist_order_items_dataset",
    "payments": "olist_order_payments_dataset",
    "reviews": "olist_order_reviews_dataset",
    "products": "olist_products_dataset",
    "sellers": "olist_sellers_dataset",
    "translation": "product_category_name_translation",
    "geolocation": "olist_geolocation_dataset"
}

# ============================================================
# 1. LEITURA EXPLÍCITA DA BRONZE
# ============================================================

dataframes_silver = {}

for chave, tabela in TABELAS.items():

    origem = f"{BRONZE_PATH}/{tabela}"

    dataframes_silver[tabela] = (
        spark.read
        .format("parquet")
        .load(origem)
    )

    print(f"Leitura concluída: {origem}")

print("Todas as tabelas foram carregadas da Bronze!")

# ============================================================
# 2. NORMALIZAÇÃO DEFENSIVA
# ============================================================

# Corrigir erros de digitação nos nomes das colunas
RENOMEAR = {
    "products": {
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length"
    }
}

# Colunas de data
DATAS = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

# Colunas numéricas
NUMERICOS = {
    "items": [
        "price",
        "freight_value",
        "order_item_id"
    ],
    "payments": [
        "payment_value",
        "payment_installments",
        "payment_sequential"
    ],
    "reviews": [
        "review_score"
    ],
    "products": [
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_photos_qty",
        "product_description_length",
        "product_name_length"
    ],
    "geolocation": [
        "geolocation_lat",
        "geolocation_lng"
    ]
}

# Colunas de CEP
CEPS = {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
    "geolocation": "geolocation_zip_code_prefix"
}

# Colunas obrigatórias por tabela
OBRIGATORIAS = {
    "orders": [
        "order_id",
        "customer_id",
        "order_status"
    ] + DATAS["orders"],

    "customers": [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ],

    "items": [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value"
    ],

    "payments": [
        "order_id",
        "payment_type",
        "payment_installments",
        "payment_value"
    ],

    "reviews": [
        "order_id",
        "review_score"
    ],

    "products": [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],

    "sellers": [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ],

    "translation": [
        "product_category_name",
        "product_category_name_english"
    ],

    "geolocation": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
}


def normalizar(chave, df):

    # 1. Corrigir nomes de colunas
    for antigo, novo in RENOMEAR.get(chave, {}).items():

        if antigo in df.columns and novo not in df.columns:
            df = df.withColumnRenamed(antigo, novo)

    # 2. Verificar colunas obrigatórias
    faltando = [
        coluna
        for coluna in OBRIGATORIAS[chave]
        if coluna not in df.columns
    ]

    assert not faltando, (
        f"{TABELAS[chave]}: "
        f"colunas ausentes na Silver: {faltando}"
    )

    # 3. Converter datas para TIMESTAMP
    for coluna in DATAS.get(chave, []):

        if coluna in df.columns:

            df = df.withColumn(
                coluna,
                F.expr(
                    f"try_cast(`{coluna}` AS TIMESTAMP)"
                )
            )

    # 4. Converter colunas numéricas para DOUBLE
    for coluna in NUMERICOS.get(chave, []):

        if coluna in df.columns:

            df = df.withColumn(
                coluna,
                F.expr(
                    f"try_cast(`{coluna}` AS DOUBLE)"
                )
            )

    # 5. Padronizar CEPs para cinco dígitos
    if chave in CEPS:

        coluna = CEPS[chave]

        cep = F.regexp_replace(
            F.trim(F.col(coluna).cast("string")),
            r"[^0-9]",
            ""
        )

        df = df.withColumn(
            coluna,
            F.when(
                cep.isNull() | (cep == ""),
                F.lit(None).cast("string")
            ).when(
                F.length(cep) < 5,
                F.lpad(cep, 5, "0")
            ).otherwise(cep)
        )

    return df


# Aplicar normalização em todas as tabelas
for chave, tabela in TABELAS.items():

    dataframes_silver[tabela] = normalizar(
        chave,
        dataframes_silver[tabela]
    )

print("Normalização concluída!")


# ============================================================
# 3. REGRAS DE QUALIDADE DOS DADOS
# ============================================================

# Regra 0: remover reviews sem informações essenciais.
# review_answer_timestamp pode ser nulo e não é obrigatório.

dataframes_silver["olist_order_reviews_dataset"] = (
    dataframes_silver["olist_order_reviews_dataset"]
    .filter(
        F.col("review_creation_date").isNotNull()
        & F.col("review_id").isNotNull()
        & F.col("order_id").isNotNull()
    )
)


# Regra 1: remover pedidos entregues sem data de entrega

dataframes_silver["olist_orders_dataset"] = (
    dataframes_silver["olist_orders_dataset"]
    .filter(
        ~(
            (F.col("order_status") == "delivered")
            & F.col("order_delivered_customer_date").isNull()
        )
    )
)


# Regra 2: remover produtos sem categoria

dataframes_silver["olist_products_dataset"] = (
    dataframes_silver["olist_products_dataset"]
    .filter(
        F.col("product_category_name").isNotNull()
        & (
            F.trim(F.col("product_category_name")) != ""
        )
    )
)


# ============================================================
# 4. DEDUPLICAÇÃO DAS AVALIAÇÕES
# ============================================================

# Manter uma avaliação por pedido:
# priorizar a resposta mais recente.
# Em caso de empate ou ausência da resposta,
# utilizar a data de criação.

reviews = dataframes_silver[
    "olist_order_reviews_dataset"
]

w_rev = Window.partitionBy("order_id").orderBy(
    F.col("review_answer_timestamp").desc_nulls_last(),
    F.col("review_creation_date").desc_nulls_last()
)

dataframes_silver["olist_order_reviews_dataset"] = (
    reviews
    .withColumn(
        "_rn",
        F.row_number().over(w_rev)
    )
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)


# ============================================================
# 5. VALIDAÇÃO DAS CHAVES PRIMÁRIAS
# ============================================================

CHAVES_PRIMARIAS = {
    "olist_orders_dataset": ["order_id"],
    "olist_customers_dataset": ["customer_id"],
    "olist_products_dataset": ["product_id"],
    "olist_sellers_dataset": ["seller_id"],
    "olist_order_items_dataset": [
        "order_id",
        "order_item_id"
    ],
    "olist_order_reviews_dataset": ["order_id"]
}

for tabela, pk in CHAVES_PRIMARIAS.items():

    df = dataframes_silver[tabela]

    # Verificar chaves nulas
    nulos = df.filter(
        F.expr(
            " OR ".join(
                [f"`{coluna}` IS NULL" for coluna in pk]
            )
        )
    ).count()

    assert nulos == 0, (
        f"{tabela}: {nulos} linhas com chave primária nula"
    )

    # Verificar duplicidades
    duplicados = (
        df.count()
        - df.select(*pk).distinct().count()
    )

    assert duplicados == 0, (
        f"{tabela}: {duplicados} linhas "
        f"com chave {pk} duplicada"
    )

print("Checagem de chaves primárias concluída!")


# ============================================================
# 6. RESUMO DA SILVER
# ============================================================

for tabela, df in dataframes_silver.items():

    print(
        f"{tabela}: {df.count()} registros | "
        f"{len(df.columns)} colunas"
    )

print("Pipeline Silver finalizado com sucesso!")


In [0]:
for table, df in dataframes_silver.items():

    output_path = f"{SILVER_PATH}/{table}"

    df.write \
        .format("parquet") \
        .mode("overwrite") \
        .save(output_path)

    print(f"Tabela {table} gravada com sucesso!")

In [0]:
for table in dataframes_silver.keys():

    df_check = spark.read.parquet(
        f"{SILVER_PATH}/{table}"
    )

    print(
        f"{table}: {df_check.count()} registros"
    )